<a href="https://colab.research.google.com/github/24127226/16-FIM-DATA-MINING/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproducible Execution Notebook: Frequent Itemset Mining Using Eclat (Julia)

This notebook provides an end-to-end reproducible pipeline for verifying correctness, benchmarking performance, and reproducing the real-world applications of the `EclatFIM` Julia package on Google Colab.

- **Course:** CSC14004 -- Data Mining and Applications (Class `24KHMT1`, Semester 1, Academic Year 2026--2027), Faculty of Information Technology, University of Science, VNU-HCM.
- **Algorithm:** Eclat (M. J. Zaki, *IEEE Transactions on Knowledge and Data Engineering*, vol. 12, no. 3, pp. 372--390, 2000).
- **Authors:** Phạm Trần Anh Quân (`24127226` -- Core Algorithm, Optimization, Application 2), Nguyễn Tiến Cường (`24127337` -- SPMF I/O, Verification, Benchmarking Infrastructure, Application 1).
- **Execution Mode:** `MODE = "quick"` executes a representative subset of the parameter grid within the free Colab session budget; `MODE = "full"` executes the complete experimental grid.

Each section is self-contained to ensure modular verification and reproducibility.

In [1]:
# Reproducibility configuration
import os

MODE = "quick"  # Valid options: "quick" or "full"
REPO_URL = "https://github.com/24127226/16-FIM-DATA-MINING.git"
REPO_BRANCH = "main"  # Target branch or release tag (e.g., "v1.0")
REPO_DIR = "/content/16-FIM-DATA-MINING"
JULIA_VERSION = "1.11.4"
os.environ["FIM_DATA_BASE"] = "https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main"

print(f"Mode: {MODE} | Repository: {REPO_URL} ({REPO_BRANCH})")

Mode: quick | Repository: https://github.com/24127226/16-FIM-DATA-MINING.git (main)


## Section 1. Environment Provisioning (Julia 1.11.4, Java, and SPMF)

Clones the repository, installs the official Julia `1.11.4` runtime, instantiates locked dependencies from `Manifest.toml`, verifies the Java runtime, downloads `spmf.jar` with SHA-256 verification, and records hardware specifications.

In [2]:
import os, shutil, subprocess

# Clone repository if not already present
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Install Julia 1.11.4
julia_bin = f"/usr/local/julia-{JULIA_VERSION}/bin/julia"
if not os.path.exists(julia_bin) and shutil.which("julia") is None:
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    subprocess.run(f"curl -fsSL {url} | tar -xz -C /usr/local", shell=True, check=True)
    subprocess.run(["ln", "-sf", julia_bin, "/usr/local/bin/julia"], check=True)

!julia --version
!julia --project=. -e "using Pkg; Pkg.instantiate(); Pkg.precompile()"

julia version 1.12.6
┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.11.4). Unexpected behavior may occur.
└ @ /content/16-FIM-DATA-MINING/Manifest.toml:0
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API /usr/local/share/julia/stdlib/v1.12/Pkg/src/API.jl:1322
   Installed DataStructures ─ v0.19.7
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2
  ◓ DataStructures
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2
  ◑ DataStructures
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2
  ◒ DataStructures
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2
  ◐ DataStructures
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2
  ◓ DataStructures
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Verify Java runtime, download SPMF reference binary, and record machine metadata
!java -version || true
!julia --project=. scripts/fetch_spmf.jl
!julia --project=. scripts/env_info.jl

[0.010s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.010s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
↓ tải https://www.philippe-fournier-viger.com/spmf/spmf.jar
! spmf.jar khác bản nhóm dùng (SHA-256 = bb371b99b3bfd58fbce6b92e0ea3ccda5f9d1fd173bfe8e40f177672117159ba).
  Trang SPMF có thể đã cập nhật phiên bản mới. Kết quả vẫn phải khớp,
  nhưng hãy ghi SHA-256 mới vào results/ENV.md khi đo hiệu năng.
✗ Java 21 quá cũ: bản spmf.jar này cần Java ≥ 25.
Đã ghi /content/16-FIM-DATA-MINING/results/ENV.md


## Section 2. Benchmark Dataset Acquisition and SHA-256 Verification

Downloads standard FIM benchmark datasets (`chess`, `mushroom`, `retail`, `accidents`, `T10I4D100K`) from the public Hugging Face repository (no authentication required) and verifies SHA-256 checksums.

In [4]:
import os
os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl benchmark

== benchmark
  ↓ tải data/benchmark/chess.txt từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/chess.txt
  ✓ SHA-256 khớp
  ↓ tải data/benchmark/mushroom.txt từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/mushroom.txt
  ✓ SHA-256 khớp
  ↓ tải data/benchmark/retail.txt từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/retail.txt
  ✓ SHA-256 khớp
  ↓ tải data/benchmark/accidents.txt từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/accidents.txt
  ✓ SHA-256 khớp
  ↓ tải data/benchmark/T10I4D100K.txt từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/T10I4D100K.txt
  ✓ SHA-256 khớp
Xong.


## Section 3. Command-Line Interface Verification on Canonical Toy Datasets

Executes `demo.jl` on `pasquier99.txt` at `minsup = 40%` (expected: 15 frequent itemsets) and `bookstore.txt` at `minsup = 0.5` (expected: 19 frequent itemsets with internal `EclatStats` counters).

In [5]:
import os
os.chdir(REPO_DIR)
!julia --project=. demo.jl --minsup 40% -o - data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 0.5 --stats -o output_bookstore.txt data/toy/bookstore.txt

1 #SUP: 3
1 2 #SUP: 2
1 2 3 #SUP: 2
1 2 3 5 #SUP: 2
1 2 5 #SUP: 2
1 3 #SUP: 3
1 3 5 #SUP: 2
1 5 #SUP: 2
2 #SUP: 4
2 3 #SUP: 3
2 3 5 #SUP: 3
2 5 #SUP: 4
3 #SUP: 4
3 5 #SUP: 3
5 #SUP: 4
=== EclatFIM (v6) ===
  Input        : data/toy/pasquier99.txt
  Giao dịch (N): 5 (m = 5 item)
  Minsup       : 40% (σ_min = 2)
  Số tập phổ biến: 15
  Thời gian đọc: 0.0001 s
  Khai thác+ghi: 0.8415 s
  Tổng thời gian: 0.8416 s
  Output       : stdout
=== EclatFIM (v6) ===
  Input        : data/toy/bookstore.txt
  Giao dịch (N): 6 (m = 5 item)
  Minsup       : 0.5 (σ_min = 3)
  Số tập phổ biến: 19
  Thời gian đọc: 0.0001 s
  Khai thác+ghi: 0.7937 s
  Tổng thời gian: 0.7938 s
  Output       : output_bookstore.txt
--- Bộ đếm nội bộ (EclatStats) ---
  Số phép giao : 13 (không đạt ngưỡng: 0)
  Số lớp con   : 9
  Độ sâu tối đa: 4
  Tidset đỉnh  : 88 bytes
  Thời gian mức 2: 0.0 s


## Section 4. Three-Tier Automated Test Suite (`test/runtests.jl`)

Executes the complete automated verification suite: SPMF I/O and `minsup_count` boundary tests, 200-database brute-force oracle verification, canonical datasets, edge cases, association rule tests (`rules.jl`), and cross-variant benchmark checks.

In [6]:
import os
os.chdir(REPO_DIR)
!julia --project=. test/runtests.jl

[ Info: Đã kiểm tra thống kê & đồng nhất giữa các biến thể trên chess; bỏ qua đối chiếu SPMF (chưa có spmf.jar/Java ≥ 25).
[ Info: Đã kiểm tra thống kê & đồng nhất giữa các biến thể trên mushroom; bỏ qua đối chiếu SPMF (chưa có spmf.jar/Java ≥ 25).
[ Info: Đã kiểm tra thống kê & đồng nhất giữa các biến thể trên retail; bỏ qua đối chiếu SPMF (chưa có spmf.jar/Java ≥ 25).
[ Info: Đã kiểm tra thống kê & đồng nhất giữa các biến thể trên T10I4D100K; bỏ qua đối chiếu SPMF (chưa có spmf.jar/Java ≥ 25).
[ Info: Bộ nhớ cấp phát: v0 = 9.92 MiB, v2 = 0.27 MiB
Test Summary: |  Pass  Total   Time
EclatFIM      | 10458  10458  11.4s


## Section 5. Chapter 2 -- Experimental Performance Evaluation

Executes `scripts/bench.jl` under the isolated-process measurement protocol with warm-up and exports results to the standardized 17-column CSV schema.

In [7]:
import os
os.chdir(REPO_DIR)
mode_arg = "quick" if MODE == "quick" else "full"
repeats = 1 if MODE == "quick" else 3
!julia --project=. scripts/bench.jl --mode {mode_arg} --variants v0,v6 --repeats {repeats} --out results/bench_colab.csv
!head -n 20 results/bench_colab.csv

→ Julia [v0] chess minsup=95% (lần 1/1)...
→ Julia [v6] chess minsup=95% (lần 1/1)...
→ Julia [v0] chess minsup=90% (lần 1/1)...
→ Julia [v6] chess minsup=90% (lần 1/1)...
→ Julia [v0] chess minsup=85% (lần 1/1)...
→ Julia [v6] chess minsup=85% (lần 1/1)...
→ Julia [v0] mushroom minsup=70% (lần 1/1)...
→ Julia [v6] mushroom minsup=70% (lần 1/1)...
→ Julia [v0] mushroom minsup=55% (lần 1/1)...
→ Julia [v6] mushroom minsup=55% (lần 1/1)...
→ Julia [v0] mushroom minsup=45% (lần 1/1)...
→ Julia [v6] mushroom minsup=45% (lần 1/1)...
→ Julia [v0] retail minsup=10% (lần 1/1)...
→ Julia [v6] retail minsup=10% (lần 1/1)...
→ Julia [v0] retail minsup=5% (lần 1/1)...
→ Julia [v6] retail minsup=5% (lần 1/1)...
→ Julia [v0] retail minsup=2% (lần 1/1)...
→ Julia [v6] retail minsup=2% (lần 1/1)...
→ Julia [v0] T10I4D100K minsup=5% (lần 1/1)...
→ Julia [v6] T10I4D100K minsup=5% (lần 1/1)...
→ Julia [v0] T10I4D100K minsup=3% (lần 1/1)...
→ Julia [v6] T10I4D100K minsup=3% (lần 1/1)...
→ Julia [v0] T10I4

## Section 6. Chapter 3 -- Application 1: Market Basket Analysis on Groceries

Reproduces the full Application 1 pipeline on `groceries.csv` (9,835 transactions, 169 items): quantitative comparison of two data transformation schemes (Table B08), threshold sensitivity grid, top-10 association rules ranked by lift (Table B09), and out-of-sample 80/20 stability evaluation (Table B10).

In [8]:
import os
from IPython.display import Markdown, display

os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl groceries
!julia --project=. apps/groceries/run_groceries.jl

with open("results/groceries_summary.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

== groceries
  ↓ tải data/apps/groceries/groceries.csv từ https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main/groceries.csv
  ✓ SHA-256 khớp
Xong.
1. Chuẩn bị dữ liệu Groceries (2 phương án biến đổi)...
2. So sánh 2 phương án biến đổi dữ liệu (B08)...
3. Khảo sát độ nhạy lưới minsup × minconf...
4. Trích xuất toàn bộ luật và top-10 theo lift (B09, F11)...
5. Kiểm tra độ ổn định luật trên chia tập 80/20 (seed=42, B10)...
Hoàn tất thực nghiệm Ứng dụng 1. Kết quả tổng hợp lưu tại: /content/16-FIM-DATA-MINING/scripts/../results/groceries_summary.md


# Kết quả Ứng dụng 1: Phân tích giỏ hàng (Groceries)

Sinh bởi `apps/groceries/run_groceries.jl` ngày 2026-11-28 bằng `EclatFIM` (`v1`, seed = 42).

## 1. So sánh phương án biến đổi dữ liệu (Bảng B08)

| Phương án | N | m | L̄ | Mật độ ρ | Tập phổ biến (1%) | Luật (conf ≥ 0,3) | Thời gian (s) |
|---|---:|---:|---:|---:|---:|---:|---:|
| 169 mặt hàng gốc (chọn) | 9835 | 169 | 4.41 | 2.61% | 333 | 125 | 0.0143 |
| Gộp 12 nhóm hàng cấp cao | 9835 | 12 | 3.12 | 25.98% | 477 | 2232 | 0.0037 |

- Phương án 169 mặt hàng gốc có mật độ **2.61%**, tiệm cận ngưỡng cân bằng lý thuyết `1/32 ≈ 3,125%` giữa biểu diễn `Vector` và `BitVector` của Eclat.
- Phương án gom 12 nhóm hàng làm mật độ tăng lên **25.98%**, dẫn đến gia tăng mạnh số tập phổ biến và số luật nhưng làm giảm tính cụ thể của luật.

## 2. Khảo sát độ nhạy trên lưới ngưỡng

| minsup | σ_min | Tập phổ biến | minconf | Số luật | Lift lớn nhất |
|---|---:|---:|---:|---:|---:|
| 0.5% | 50 | 1001 | 0.2 | 893 | 5.212 |
| 0.5% | 50 | 1001 | 0.3 | 487 | 4.716 |
| 0.5% | 50 | 1001 | 0.5 | 120 | 3.691 |
| 1% | 99 | 333 | 0.2 | 234 | 3.295 |
| 1% | 99 | 333 | 0.3 | 125 | 3.295 |
| 1% | 99 | 333 | 0.5 | 15 | 3.03 |
| 2% | 197 | 122 | 0.2 | 73 | 2.842 |
| 2% | 197 | 122 | 0.3 | 37 | 2.842 |
| 2% | 197 | 122 | 0.5 | 1 | 2.007 |

- Cặp ngưỡng **minsup = 1%, minconf = 0.3** được lựa chọn vì tạo ra **125 luật** (thuộc vùng mục tiêu 50–300 luật), cân bằng giữa độ bao phủ và độ tin cậy diễn giải.

## 3. Mười luật kết hợp dẫn đầu theo độ nâng (Bảng B09)

| Hạng | Tiền đề (X) | Hệ quả (Y) | σ | Support | Confidence | Lift |
|---:|---|---|---:|---:|---:|---:|
| 1 | citrus fruit + other vegetables | root vegetables | 102 | 0.0104 | 0.3592 | 3.295 |
| 2 | other vegetables + tropical fruit | root vegetables | 121 | 0.0123 | 0.3428 | 3.145 |
| 3 | beef | root vegetables | 171 | 0.0174 | 0.3314 | 3.04 |
| 4 | citrus fruit + root vegetables | other vegetables | 102 | 0.0104 | 0.5862 | 3.03 |
| 5 | root vegetables + tropical fruit | other vegetables | 121 | 0.0123 | 0.5845 | 3.021 |
| 6 | other vegetables + whole milk | root vegetables | 228 | 0.0232 | 0.3098 | 2.842 |
| 7 | curd + whole milk | yogurt | 99 | 0.0101 | 0.3852 | 2.761 |
| 8 | rolls/buns + root vegetables | other vegetables | 120 | 0.0122 | 0.5021 | 2.595 |
| 9 | root vegetables + yogurt | other vegetables | 127 | 0.0129 | 0.5 | 2.584 |
| 10 | tropical fruit + whole milk | yogurt | 149 | 0.0151 | 0.3582 | 2.568 |

- **Phân tích tiêu chí xếp hạng:** mặt hàng `whole milk` xuất hiện trong **2513/9835 hóa đơn (25.55%)**, khiến các luật đạt confidence cao nhất chủ yếu mang hệ quả `whole milk` nhưng chỉ đạt lift trung bình (1,8–2,3):
  - `citrus fruit + root vegetables ⇒ other vegetables`: conf = 0.586, lift = 3.03
  - `root vegetables + tropical fruit ⇒ other vegetables`: conf = 0.585, lift = 3.021
  - `curd + yogurt ⇒ whole milk`: conf = 0.582, lift = 2.279
  - `butter + other vegetables ⇒ whole milk`: conf = 0.574, lift = 2.245
  - `root vegetables + tropical fruit ⇒ whole milk`: conf = 0.57, lift = 2.231

## 4. Đánh giá độ ổn định trên phép chia 80/20 (Bảng B10, train=7868, test=1967)

| Hạng | Luật (X ⇒ Y) | Train Lift | Test Lift | Biến thiên | Train Conf | Test Conf |
|---:|---|---:|---:|---:|---:|---:|
| 1 | citrus fruit + other vegetables ⇒ root vegetables | 3.325 | 3.175 | -4.5% | 0.361 | 0.352 |
| 2 | other vegetables + tropical fruit ⇒ root vegetables | 3.242 | 2.734 | -15.7% | 0.352 | 0.303 |
| 3 | citrus fruit + root vegetables ⇒ other vegetables | 3.159 | 2.568 | -18.7% | 0.61 | 0.5 |
| 4 | beef ⇒ root vegetables | 3.093 | 2.831 | -8.5% | 0.336 | 0.314 |
| 5 | root vegetables + tropical fruit ⇒ other vegetables | 3.04 | 2.935 | -3.4% | 0.587 | 0.571 |
| 6 | whipped/sour cream + whole milk ⇒ root vegetables | 2.925 | 1.805 | -38.3% | 0.317 | 0.2 |
| 7 | other vegetables + yogurt ⇒ root vegetables | 2.864 | 2.203 | -23.1% | 0.311 | 0.244 |
| 8 | other vegetables + whole milk ⇒ root vegetables | 2.787 | 3.072 | 10.2% | 0.303 | 0.34 |
| 9 | other vegetables + whipped/sour cream ⇒ yogurt | 2.612 | 2.214 | -15.3% | 0.367 | 0.302 |
| 10 | rolls/buns + root vegetables ⇒ other vegetables | 2.588 | 2.628 | 1.5% | 0.5 | 0.512 |

- Toàn bộ **10/10** luật duy trì `test_lift > 1,5` trên tập kiểm tra độc lập 20%.


## Section 7. Chapter 3 -- Application 2: Visual Pattern Discovery via Bag of Visual Words

Execution cells for Application 2 on the Caltech-101 image dataset will be integrated upon completion of `apps/bovw/`.

## Section 8. Report and Artifacts

The complete project report is stored at `docs/Report.pdf` in this repository.